# Anomaly Prediction in Mask Data
**Goal:** given an image of a product, predict a **pixel-level anomaly mask** that shows exactly where a defect is.

**How this differs from the usual approach (supervised YOLO boxes):**
- **Unsupervised:** the model only ever sees *normal* images. It learns what "good" looks like, so it can flag defects it has never seen.
- **Masks, not boxes:** every pixel gets an anomaly score, which we turn into a heatmap and a binary mask.
- **Honest thresholding:** the mask threshold comes from held-out *normal* images (deployable). The best-possible "oracle" threshold is reported only for reference.
- **Method:** a PatchCore-style memory bank written from scratch in PyTorch (pretrained WideResNet-50 features, greedy coreset, nearest-neighbour scoring).

**Dataset:** MVTec AD (CC BY-NC-SA 4.0). It has normal training images and test images with ground-truth defect masks.

> Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
# 1. Setup
import torch, platform
print("PyTorch", torch.__version__, "| Python", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU!")
!pip -q install huggingface_hub

In [ ]:
# 2. Configuration - change these to run your own experiments
CATEGORIES   = ["bottle", "hazelnut", "screw", "carpet", "leather"]  # objects + textures
DATA_ROOT    = "/content/mvtec"
RESIZE, CROP = 256, 224
CORESET_RATIO = 0.10     # fraction of patch features kept in the memory bank
CALIB_FRACTION = 0.10    # share of normal train images held out to pick the mask threshold
CALIB_QUANTILE = 0.999   # pixel-score quantile on normal images used as the threshold
SEED = 42

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 3. Download only the chosen MVTec AD categories (mirror with the original folder layout)
from huggingface_hub import snapshot_download
snapshot_download(repo_id="foersben/mvtec-ad", repo_type="dataset",
                  allow_patterns=[f"{c}/*" for c in CATEGORIES], local_dir=DATA_ROOT)
import os
for c in CATEGORIES:
    n_train = len(os.listdir(f"{DATA_ROOT}/{c}/train/good"))
    defects = sorted(os.listdir(f"{DATA_ROOT}/{c}/test"))
    print(f"{c:10s} train(good)={n_train:4d}  test types={defects}")

In [ ]:
# 4. Dataset: image + ground-truth mask (all-zero mask for good images)
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

img_tf  = T.Compose([T.Resize(RESIZE), T.CenterCrop(CROP), T.ToTensor(),
                     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
mask_tf = T.Compose([T.Resize(RESIZE, interpolation=T.InterpolationMode.NEAREST),
                     T.CenterCrop(CROP), T.ToTensor()])

class MVTecSplit(Dataset):
    def __init__(self, items):  # items: list of (image_path, mask_path or None, label)
        self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, m, y = self.items[i]
        img = img_tf(Image.open(p).convert("RGB"))
        mask = (mask_tf(Image.open(m).convert("L")) > 0.5).float() if m else torch.zeros(1, CROP, CROP)
        return img, mask, y, str(p)

def build_splits(cat):
    root = Path(DATA_ROOT) / cat
    good = sorted((root / "train" / "good").glob("*.png"))
    rng = random.Random(SEED); rng.shuffle(good)
    n_cal = max(1, int(len(good) * CALIB_FRACTION))
    calib, train = good[:n_cal], good[n_cal:]
    test = []
    for d in sorted((root / "test").iterdir()):
        for p in sorted(d.glob("*.png")):
            if d.name == "good":
                test.append((p, None, 0))
            else:
                test.append((p, root / "ground_truth" / d.name / f"{p.stem}_mask.png", 1))
    wrap = lambda ps: [(p, None, 0) for p in ps]
    return MVTecSplit(wrap(train)), MVTecSplit(wrap(calib)), MVTecSplit(test)

In [ ]:
# 5. Feature extractor: mid-level WideResNet-50 features (layer2 + layer3), locally averaged
import torch.nn as nn, torch.nn.functional as F
from torchvision.models import wide_resnet50_2, Wide_ResNet50_2_Weights

class PatchFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        net = wide_resnet50_2(weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1).eval()
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool, net.layer1)
        self.l2, self.l3 = net.layer2, net.layer3
    @torch.no_grad()
    def forward(self, x):
        f2 = self.l2(self.stem(x))
        f3 = self.l3(f2)
        f2 = F.avg_pool2d(f2, 3, 1, 1)           # neighbourhood context
        f3 = F.avg_pool2d(f3, 3, 1, 1)
        f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([f2, f3], 1)            # B x 1536 x 28 x 28

extractor = PatchFeatures().to(DEVICE)

@torch.no_grad()
def extract(ds, bs=32):
    feats, masks, labels, paths = [], [], [], []
    for x, m, y, p in DataLoader(ds, batch_size=bs, num_workers=2):
        feats.append(extractor(x.to(DEVICE))); masks.append(m); labels.append(y); paths += p
    return torch.cat(feats), torch.cat(masks), torch.cat(labels), paths

In [ ]:
# 6. Memory bank with greedy coreset + nearest-neighbour anomaly maps
from scipy.ndimage import gaussian_filter

def greedy_coreset(bank, ratio, proj_dim=128):
    n = bank.shape[0]; k = max(1, int(n * ratio))
    g = torch.Generator(device=bank.device).manual_seed(SEED)
    z = bank @ torch.randn(bank.shape[1], proj_dim, device=bank.device, generator=g)  # random projection for speed
    idx = [0]
    min_d = ((z - z[0]) ** 2).sum(1)
    for _ in range(k - 1):
        i = int(torch.argmax(min_d)); idx.append(i)
        min_d = torch.minimum(min_d, ((z - z[i]) ** 2).sum(1))
    return bank[idx]

def to_patches(f):  # B x C x H x W -> (B*H*W) x C
    return f.permute(0, 2, 3, 1).reshape(-1, f.shape[1])

@torch.no_grad()
def anomaly_maps(feats, bank, chunk=8):
    B, C, H, W = feats.shape
    out = []
    for i in range(0, B, chunk):
        f = to_patches(feats[i:i + chunk])
        d = torch.cdist(f, bank).min(1).values.reshape(-1, 1, H, W)
        d = F.interpolate(d, size=(CROP, CROP), mode="bilinear", align_corners=False)
        out.append(d.cpu())
    maps = torch.cat(out)[:, 0].numpy()
    return np.stack([gaussian_filter(m, sigma=4) for m in maps])

In [ ]:
# 7. Train (= build memory bank) and evaluate every category
import time, pandas as pd
from sklearn.metrics import roc_auc_score, precision_recall_curve

def mask_scores(pred, gt):
    inter = (pred & gt).sum((1, 2)); union = (pred | gt).sum((1, 2))
    dice = 2 * inter / (pred.sum((1, 2)) + gt.sum((1, 2)) + 1e-8)
    return float(np.mean(inter / (union + 1e-8))), float(np.mean(dice))

results, store = [], {}
for cat in CATEGORIES:
    t0 = time.time()
    train_ds, calib_ds, test_ds = build_splits(cat)
    tr_f, *_ = extract(train_ds)
    bank = greedy_coreset(to_patches(tr_f), CORESET_RATIO); del tr_f
    fit_s = time.time() - t0

    cal_f, *_ = extract(calib_ds)
    thr = float(np.quantile(anomaly_maps(cal_f, bank), CALIB_QUANTILE))  # threshold from normal images only

    te_f, te_m, te_y, te_p = extract(test_ds)
    maps = anomaly_maps(te_f, bank)
    gt = te_m[:, 0].numpy().astype(bool); y = te_y.numpy()
    img_score = maps.reshape(len(maps), -1).max(1)

    prec, rec, ths = precision_recall_curve(gt.ravel(), maps.ravel())
    f1 = 2 * prec * rec / (prec + rec + 1e-8); oracle_thr = float(ths[np.argmax(f1[:-1])])

    bad = y == 1
    iou_cal, dice_cal = mask_scores(maps[bad] > thr, gt[bad])
    iou_orc, dice_orc = mask_scores(maps[bad] > oracle_thr, gt[bad])
    results.append(dict(category=cat, image_AUROC=roc_auc_score(y, img_score),
                        pixel_AUROC=roc_auc_score(gt.ravel(), maps.ravel()),
                        IoU=iou_cal, Dice=dice_cal, IoU_oracle=iou_orc, Dice_oracle=dice_orc,
                        bank_size=len(bank), fit_seconds=round(fit_s, 1)))
    store[cat] = dict(maps=maps, gt=gt, y=y, paths=te_p, thr=thr)
    print(f"{cat:10s} done in {time.time() - t0:5.1f}s")
    del bank, te_f; torch.cuda.empty_cache()

df = pd.DataFrame(results).set_index("category")
df.loc["mean"] = df.mean(numeric_only=True)
df.round(3)

In [ ]:
# 8. Visualise: image | ground-truth mask | heatmap | predicted mask
import matplotlib.pyplot as plt

def show_examples(cat, n=3):
    s = store[cat]; bad = np.where(s["y"] == 1)[0]
    pick = np.random.default_rng(SEED).choice(bad, size=min(n, len(bad)), replace=False)
    vmax = np.percentile(s["maps"], 99.9)
    fig, ax = plt.subplots(len(pick), 4, figsize=(12, 3 * len(pick)))
    for r, i in enumerate(pick):
        img = T.CenterCrop(CROP)(T.Resize(RESIZE)(Image.open(s["paths"][i]).convert("RGB")))
        defect = Path(s["paths"][i]).parent.name
        ax[r, 0].imshow(img); ax[r, 0].set_title(f"{cat} / {defect}")
        ax[r, 1].imshow(s["gt"][i], cmap="gray"); ax[r, 1].set_title("ground truth")
        ax[r, 2].imshow(img); ax[r, 2].imshow(s["maps"][i], cmap="jet", alpha=0.5, vmin=0, vmax=vmax)
        ax[r, 2].set_title("anomaly heatmap")
        ax[r, 3].imshow(img); ax[r, 3].contour(s["maps"][i] > s["thr"], colors="red", linewidths=1.5)
        ax[r, 3].set_title("predicted mask")
        for a in ax[r]: a.axis("off")
    plt.tight_layout(); plt.show()

for c in CATEGORIES:
    show_examples(c)

In [ ]:
# 9. Summary chart
ax = df.drop("mean")[["image_AUROC", "pixel_AUROC", "Dice", "Dice_oracle"]].plot.bar(figsize=(10, 4), rot=0)
ax.set_ylim(0, 1.05); ax.set_title("Anomaly mask prediction per category"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# 10. (Optional) Save results to Google Drive so they survive a runtime reset
# from google.colab import drive; drive.mount("/content/drive")
# os.makedirs("/content/drive/MyDrive/anomaly_mask_project", exist_ok=True)
# df.to_csv("/content/drive/MyDrive/anomaly_mask_project/results.csv")